# **Запросы для аггрегации данных**

In [4]:
import os
import pandas as pd
from datetime import datetime
from sqlalchemy import (create_engine, String, Integer, Float, DateTime,
                        select, func, case, desc, text)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

In [5]:
class Base(DeclarativeBase):
    pass

class Car(Base):
    __tablename__ = "cars"
    listing_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    brand: Mapped[str] = mapped_column(String(50))
    model: Mapped[str] = mapped_column(String(50))
    year: Mapped[int] = mapped_column(Integer)
    price: Mapped[int] = mapped_column(Integer)
    volume_l: Mapped[float | None] = mapped_column(Float)
    power_hp: Mapped[float | None] = mapped_column(Float)
    fuel: Mapped[str | None] = mapped_column(String(20))
    transmission: Mapped[str | None] = mapped_column(String(20))
    drive: Mapped[str | None] = mapped_column(String(20))
    mileage_km: Mapped[float | None] = mapped_column(Float)
    url: Mapped[str] = mapped_column(String(200))
    scraped_at: Mapped[datetime] = mapped_column(DateTime)

In [10]:
engine = create_engine("sqlite:////content/drom.db")

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM cars")).scalar())

1000


Топ-10 марок по числу объявлений

In [11]:
q1 = (
    select(
        Car.brand,
        func.count().label("cnt"),
        func.round(func.avg(Car.price)).label("avg_price"),
        func.round(func.avg(Car.mileage_km)).label("avg_mileage"),
    )
    .group_by(Car.brand)
    .having(func.count() >= 10)
    .order_by(desc("cnt"))
    .limit(10)
)
df1 = pd.read_sql_query(q1, engine)
df1

,brand,cnt,avg_price,avg_mileage
0,lada,74,527884.0,120924.0
1,toyota,71,1609549.0,209992.0
2,volkswagen,66,1495566.0,182640.0
3,hyundai,58,1445284.0,166788.0
4,kia,56,1196625.0,167875.0
5,mercedes-benz,45,3602709.0,183782.0
6,mitsubishi,44,1146563.0,216031.0
7,nissan,41,1062876.0,189168.0
8,ford,41,702097.0,190813.0
9,skoda,40,1365796.0,161266.0


Цена и пробег по возрастным группам

In [12]:
age_group = case(
    (Car.year < 2010, "до 2010"),
    (Car.year < 2015, "2010-2014"),
    (Car.year < 2020, "2015-2019"),
    else_="2020 и новее",
).label("age_group")

q2 = (
    select(
        age_group,
        func.count().label("cnt"),
        func.round(func.avg(Car.price)).label("avg_price"),
        func.round(func.avg(Car.mileage_km)).label("avg_mileage"),
    )
    .group_by(age_group)
    .order_by(func.min(Car.year))
)
df2 = pd.read_sql_query(q2, engine)
df2

,age_group,cnt,avg_price,avg_mileage
0,до 2010,244,543125.0,253800.0
1,2010-2014,242,959951.0,199009.0
2,2015-2019,235,2045962.0,143498.0
3,2020 и новее,279,3132295.0,63587.0


Топливо X Коробка передач

In [15]:
sql3 = """
SELECT fuel, transmission, COUNT(*) AS cnt, ROUND(AVG(price)) AS avg_price
FROM cars
WHERE fuel IS NOT NULL AND transmission IS NOT NULL
GROUP BY fuel, transmission
"""
df3 = pd.read_sql_query(text(sql3), engine)

pivot_cnt   = df3.pivot_table(index="fuel", columns="transmission", values="cnt", fill_value=0)
pivot_price = df3.pivot_table(index="fuel", columns="transmission", values="avg_price")
pivot_cnt

transmission,АКПП,вариатор,механика,робот
fuel,,,,
бензин,325.0,116.0,274.0,147.0
гибрид,18.0,8.0,0.0,6.0
дизель,50.0,0.0,16.0,3.0
электро,2.0,0.0,0.0,0.0


In [16]:
pivot_price

transmission,АКПП,вариатор,механика,робот
fuel,,,,
бензин,1995718.0,1491337.0,559051.0,2029003.0
гибрид,5838452.0,2650642.0,NaN,2300033.0
дизель,3965640.0,NaN,909186.0,1632067.0
электро,1787500.0,NaN,NaN,NaN


Цена за 1 л.с. по маркам

In [19]:
q4 = (
    select(
        Car.brand,
        func.count().label("cnt"),
        func.round(func.avg(Car.price / Car.power_hp)).label("rub_per_hp"),
    )
    .where(Car.power_hp > 0)
    .group_by(Car.brand)
    .having(func.count() >= 10)
    .order_by(desc("rub_per_hp"))
)
df4 = pd.read_sql_query(q4, engine)

df4.head(5)

,brand,cnt,rub_per_hp
0,land_rover,11,18370.0
1,porsche,11,15696.0
2,mercedes-benz,45,14066.0
3,audi,33,12657.0
4,haval,21,12238.0


In [18]:
df4.tail(5)

,brand,cnt,rub_per_hp
22,chevrolet,26,6494.0
23,lada,74,5436.0
24,ford,41,5222.0
25,opel,22,4678.0
26,peugeot,16,3528.0


Медианная цена против средней по маркам

In [20]:
df5 = pd.read_sql_query(select(Car.brand, Car.price), engine)

res5 = (
    df5.groupby("brand")["price"]
       .agg(cnt="count", median="median", mean="mean")
       .query("cnt >= 10")
       .round(0)
       .sort_values("median", ascending=False)
       .head(10)
)
res5["mean_minus_median"] = res5["mean"] - res5["median"]
res5

,cnt,median,mean,mean_minus_median
brand,,,,
porsche,11,5530000.0,6407091.0,877091.0
audi,33,3100000.0,3229471.0,129471.0
lexus,21,2599999.0,2587271.0,-12728.0
mercedes-benz,45,2450000.0,3602709.0,1152709.0
bmw,34,2349500.0,3023765.0,674265.0
mazda,21,2099700.0,1809113.0,-290587.0
land_rover,11,2067900.0,5539536.0,3471636.0
haval,21,1949900.0,2092838.0,142938.0
geely,13,1890000.0,2210869.0,320869.0
